# Variational Autoencoders: ELBO, Inference, and Learning

The **variational autoencoder** is one of the clearest examples of how a deep generative model emerges from a probabilistic problem. We start from a latent-variable model $p_\theta(\boldsymbol{x}, \boldsymbol{z}) = p_\theta(\boldsymbol{x} | \boldsymbol{z}) p(\boldsymbol{z})$, where $\boldsymbol{z}$ is a latent code and the conditional distribution $p_\theta(\boldsymbol{x} | \boldsymbol{z})$ is implemented by a neural decoder. This model is easy to sample from: first draw $\boldsymbol{z} \sim p(\boldsymbol{z})$, then draw $\boldsymbol{x} \sim p_\theta(\boldsymbol{x} | \boldsymbol{z})$. The challenge is learning, because the marginal likelihood
:::{math}
p_\theta(\boldsymbol{x}) = \int p_\theta(\boldsymbol{x} | \boldsymbol{z}) p(\boldsymbol{z}) \, d\boldsymbol{z}
:::
is usually intractable, and the exact posterior $p_\theta(\boldsymbol{z} | \boldsymbol{x})$ is rarely available in closed form.

The VAE addresses this by introducing an encoder, also called a recognition model, $q_\phi(\boldsymbol{z} | \boldsymbol{x})$, whose purpose is to approximate the true posterior. The word variational signals that learning is formulated as an optimization problem over a family of approximate posterior distributions.

This is the first chapter where all three strands of the course fully meet: neural parameterization, probabilistic latent-variable modeling, and tractable optimization. The previous chapter ended by identifying the two hard quantities in a deep latent-variable model, namely the marginal likelihood and the exact posterior. The VAE should therefore be read as the first serious answer to that bottleneck. It does not remove the intractability by magic. It reorganizes it into a lower bound that can be optimized with neural encoders and decoders.

A simple mental picture helps here. Imagine a dataset of shoes. A useful latent code might organize broad factors such as ankle boot versus sandal, heel height, or overall silhouette. The decoder then learns how those hidden factors map back into pixels. The encoder tries to reverse that map from a specific image. The VAE is the mathematical framework that makes this bidirectional story trainable.

## Derivation of the Evidence Lower Bound

The derivation begins with the identity
:::{math}
\log p_\theta(\boldsymbol{x})
=
\log \int q_\phi(\boldsymbol{z} | \boldsymbol{x})
\frac{p_\theta(\boldsymbol{x}, \boldsymbol{z})}{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\, d\boldsymbol{z}.
:::
Since the logarithm is concave, Jensen's inequality immediately yields a lower bound. This is the central theorem of the chapter.

```{prf:theorem} Evidence lower bound
:label: thm-elbo

For any density $q_\phi(\boldsymbol{z} | \boldsymbol{x})$ whose support contains the support of $p_\theta(\boldsymbol{z} | \boldsymbol{x})$,
:::{math}
\log p_\theta(\boldsymbol{x})
\geq
\mathbb{E}_{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\left[
\log p_\theta(\boldsymbol{x}, \boldsymbol{z})
-
\log q_\phi(\boldsymbol{z} | \boldsymbol{x})
\right].
:::
The right-hand side is called the evidence lower bound, or ELBO.
```

```{prf:proof}
Write
:::{math}
\log p_\theta(\boldsymbol{x})
=
\log \mathbb{E}_{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\left[
\frac{p_\theta(\boldsymbol{x}, \boldsymbol{z})}{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\right].
:::
Jensen's inequality for the concave logarithm implies
:::{math}
\log p_\theta(\boldsymbol{x})
\geq
\mathbb{E}_{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\left[
\log \frac{p_\theta(\boldsymbol{x}, \boldsymbol{z})}{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\right].
:::
Expanding the logarithm gives the stated formula.
```

The **ELBO** is more than a technical inequality. It is the training objective that makes the model practical. Expanding the joint density yields
:::{math}
\mathcal{L}_{ELBO}(\theta, \phi; \boldsymbol{x})
=
\mathbb{E}_{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
[\log p_\theta(\boldsymbol{x} | \boldsymbol{z})]
-
\operatorname{KL}(q_\phi(\boldsymbol{z} | \boldsymbol{x}) \| p(\boldsymbol{z})).
:::
The first term is a reconstruction term: it encourages the decoder to explain the observed image well from latent codes sampled from the encoder. The second term regularizes the approximate posterior toward the prior. This is what turns a deterministic autoencoder into a generative model. If the latent codes produced by the encoder remain close to a simple reference distribution, then sampling from that reference distribution at generation time becomes meaningful.

```{prf:theorem} ELBO decomposition with posterior gap
:label: thm-elbo-gap

For every observation $\boldsymbol{x}$,
:::{math}
\log p_\theta(\boldsymbol{x})
=
\mathcal{L}_{ELBO}(\theta, \phi; \boldsymbol{x})
+
\operatorname{KL}\big(q_\phi(\boldsymbol{z} | \boldsymbol{x}) \| p_\theta(\boldsymbol{z} | \boldsymbol{x})\big).
:::
```

```{prf:proof}
Starting from the ELBO expression,
:::{math}
\mathcal{L}_{ELBO}
=
\mathbb{E}_{q_\phi}
[\log p_\theta(\boldsymbol{x}, \boldsymbol{z}) - \log q_\phi(\boldsymbol{z} | \boldsymbol{x})].
:::
Add and subtract $\log p_\theta(\boldsymbol{z} | \boldsymbol{x})$. Using
:::{math}
\log p_\theta(\boldsymbol{x}, \boldsymbol{z})
=
\log p_\theta(\boldsymbol{z} | \boldsymbol{x}) + \log p_\theta(\boldsymbol{x}),
:::
we obtain
:::{math}
\mathcal{L}_{ELBO}
=
\log p_\theta(\boldsymbol{x})
-
\mathbb{E}_{q_\phi}
\left[
\log \frac{q_\phi(\boldsymbol{z} | \boldsymbol{x})}{p_\theta(\boldsymbol{z} | \boldsymbol{x})}
\right].
:::
The expectation is exactly the Kullback-Leibler divergence between the approximate and exact posterior, which proves the identity.
```

This identity explains why the bound is useful. Maximizing the ELBO simultaneously pushes upward the data log-likelihood and pushes downward the gap between the recognition model and the true posterior. The quality of inference and the quality of generation are therefore coupled. If the approximate posterior family is too restrictive, the model may learn a decoder that is easier for the encoder to support rather than one that is genuinely optimal for the data.

```{admonition} Numerical Example: Reading the ELBO Terms
:class: numerical-example

Suppose an encoder sees an image of a sneaker and outputs a two-dimensional Gaussian posterior with mean $\boldsymbol{\mu} = (1.2, -0.4)$ and standard deviations $(0.5, 0.8)$. The corresponding variational distribution says that the image is encoded near the point $(1.2, -0.4)$ in latent space, but with noticeable uncertainty, especially in the second coordinate.

If the decoder reconstructs the sneaker well, the reconstruction term in the ELBO is favorable. But the KL term also checks how far this posterior has drifted from the standard Gaussian prior. Here the first coordinate is shifted away from zero, so there is a regularization cost. The ELBO is therefore balancing two pressures at once: keep enough latent information to reconstruct the sneaker, but do not let the posterior wander arbitrarily far from the simple prior that we want to sample from at generation time.
```

There is also an important optimization lesson hidden here. The encoder $q_\phi(\boldsymbol{z} | \boldsymbol{x})$ is often called an amortized inference model because a single neural network is trained to solve approximate inference for every observation in the dataset at once. This is computationally powerful, but it also creates an additional approximation layer beyond the choice of variational family itself. The encoder may fail to represent the true posterior not only because the family is too simple, but also because the shared inference network does not allocate enough capacity to every observation equally well.

For a PhD audience, this is worth stressing because many later improvements to VAEs can be read as interventions on this exact bottleneck. Some methods enrich the variational family, some redesign the decoder likelihood, some modify the objective weighting, and some attack amortization error directly. The classical ELBO is therefore not the end of the story. It is the organizing baseline from which a whole research program begins.

## Gaussian Encoders and the Reparameterization Trick

In practice, a common choice is
:::{math}
q_\phi(\boldsymbol{z} | \boldsymbol{x}) =
\mathcal{N}(\boldsymbol{z}; \boldsymbol{\mu}_\phi(\boldsymbol{x}), \operatorname{diag}(\boldsymbol{\sigma}_\phi^2(\boldsymbol{x}))).
:::
The encoder network outputs the mean vector and the log-variance vector. If the prior is standard Gaussian, the KL term can be computed in closed form. The remaining challenge is differentiating through a stochastic sample from $q_\phi(\boldsymbol{z} | \boldsymbol{x})$. The reparameterization trick solves this by writing
:::{math}
\boldsymbol{z}
=
\boldsymbol{\mu}_\phi(\boldsymbol{x})
+
\boldsymbol{\sigma}_\phi(\boldsymbol{x}) \odot \boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}).
:::
Now the randomness is isolated in $\boldsymbol{\varepsilon}$, which does not depend on $\phi$. The map from parameters to sample becomes differentiable, and gradient-based learning becomes feasible.

```{prf:theorem} Closed-form KL for a diagonal Gaussian encoder
:label: thm-diagonal-gaussian-kl

Let
:::{math}
q_\phi(\boldsymbol{z} | \boldsymbol{x})
=
\mathcal{N}\big(
    \boldsymbol{z};
    \boldsymbol{\mu},
    \operatorname{diag}(\boldsymbol{\sigma}^2)
\big)
:::
and let the prior be
:::{math}
p(\boldsymbol{z}) = \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}).
:::
Then
:::{math}
\operatorname{KL}(q_\phi(\boldsymbol{z} | \boldsymbol{x}) \| p(\boldsymbol{z}))
=
\frac{1}{2}
\sum_{j=1}^d
\left(
    \mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1
\right).
:::
```

```{prf:proof}
By definition,
:::{math}
\operatorname{KL}(q \| p)
=
\mathbb{E}_q[\log q(\boldsymbol{z}) - \log p(\boldsymbol{z})].
:::
For the diagonal Gaussian posterior,
:::{math}
\log q(\boldsymbol{z})
=
-\frac{1}{2}
\sum_{j=1}^d
\left[
    \log(2\pi \sigma_j^2)
    +
    \frac{(z_j-\mu_j)^2}{\sigma_j^2}
\right],
:::
while for the standard Gaussian prior,
:::{math}
\log p(\boldsymbol{z})
=
-\frac{1}{2}
\sum_{j=1}^d
\left[
    \log(2\pi)
    +
    z_j^2
\right].
:::
Subtracting and taking expectation under $q$, we use
:::{math}
\mathbb{E}_q[(z_j-\mu_j)^2] = \sigma_j^2,
\qquad
\mathbb{E}_q[z_j^2] = \mu_j^2 + \sigma_j^2.
:::
The constant $\log(2\pi)$ terms cancel, leaving
:::{math}
\operatorname{KL}(q \| p)
=
\frac{1}{2}
\sum_{j=1}^d
\left(
    \mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1
\right),
:::
which proves the formula.
```

This explicit KL formula is one of the practical reasons the Gaussian VAE became such a standard teaching example. The reconstruction term must still be estimated through stochastic samples, but the regularization term is analytically available. The resulting objective is therefore simple enough to implement directly while still expressing the real variational logic of the model.

## Strengths, Limitations, and the Problem of Blurriness

VAEs are conceptually elegant, statistically grounded, and usually stable to train. They provide an explicit encoder, which makes latent interpolation and amortized inference natural. Yet they also exhibit limitations. The approximate posterior may be too simple, the decoder likelihood may be misspecified, and pixelwise Gaussian reconstruction losses often encourage averages when multiple plausible outputs exist. This is one reason generated images may appear blurry compared with samples from stronger adversarial or diffusion-based models.

The correct interpretation of this limitation is subtle. The VAE is not blurry because it is probabilistic. It is blurry because of specific modeling choices, especially the combination of simple variational families and reconstruction likelihoods that reward conditional means. Later improvements therefore act on several fronts: richer posterior families, more expressive decoders, alternative priors, hierarchical latent structures, perceptual losses, and latent-space decoders as in latent diffusion models.

```{figure} ../assets/images/VAE_architecture.png
:width: 76%
:align: center

The encoder-decoder structure of the variational autoencoder. The stochastic bottleneck is the feature that turns a deterministic autoencoder into a probabilistic latent-variable model.
```

Another limitation that deserves explicit mention is **posterior collapse**. If the decoder becomes strong enough to model the data with little dependence on the latent code, the optimizer may drive the approximate posterior toward the prior and effectively ignore $\boldsymbol{z}$. In that regime, the KL term becomes very small, but the latent representation ceases to carry meaningful information. This phenomenon is especially visible in sequence VAEs and in image models with highly expressive decoders, but the conceptual issue is general: the ELBO does not force the model to use the latent variable unless doing so helps the total objective.

Possible remedies illustrate the broader design space of VAEs. One may weaken or schedule the KL penalty, as in $\beta$-VAE style variants or KL warm-up. One may enrich the posterior family through normalizing flows or hierarchical structure. One may alter the decoder likelihood or use perceptual losses to reduce the incentive toward pixelwise averaging. One may also redesign the architecture so that the decoder cannot ignore the latent code too easily. These are not minor tweaks. They show that once the basic VAE logic is understood, a large family of principled variants becomes accessible.

The VAE should be remembered as the canonical bridge between probabilistic latent-variable modeling and neural network training. Its historical formulations were introduced in {cite}`kingma2013auto` and {cite}`rezende2014stochastic`. Many later methods, including diffusion models, can be better understood once one has internalized the VAE logic: choose a latent construction, derive a tractable surrogate objective, and use neural networks to amortize the hard parts.